# 03 — Model Comparison & Evaluation

> **Mục tiêu:** So sánh 4 models, ROC curves, confusion matrices, kết luận

---

## 1. Setup


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    f1_score, precision_score, recall_score,
    confusion_matrix
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 11

PROJECT_ROOT = Path('../..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'

# Load evaluation results
with open(PROCESSED_DIR / 'evaluation_results.json') as f:
    eval_results = json.load(f)

with open(PROCESSED_DIR / 'anomaly_results.json') as f:
    anomaly_results = json.load(f)

with open(PROCESSED_DIR / 'dl_results.json') as f:
    dl_results = json.load(f)

y_test = np.load(PROCESSED_DIR / 'y_test.npy')

print(f'Test set: {len(y_test):,} samples')
print(f'Exfil rate: {y_test.mean()*100:.3f}%')
print(f'\nModels evaluated: {len(eval_results["models"])}')
for m in eval_results['models']:
    print(f'  - {m["name"]} ({m["type"]})')


---

## 2. ROC Curves


In [ ]:
# Load and plot saved ROC curves
import matplotlib.image as mpimg

roc_path = PROJECT_ROOT / 'notebooks' / 'roc_curves.png'
if roc_path.exists():
    img = mpimg.imread(roc_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('ROC Curves — Generated by evaluate.py', fontsize=14)
    plt.show()
else:
    print('ROC curves not yet generated. Run: python src/train/evaluate.py')


---

## 3. Confusion Matrices


In [ ]:
cm_path = PROJECT_ROOT / 'notebooks' / 'confusion_matrices.png'
if cm_path.exists():
    img = mpimg.imread(cm_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrices — Generated by evaluate.py', fontsize=14)
    plt.show()
else:
    print('Confusion matrices not yet generated. Run: python src/train/evaluate.py')


---

## 4. Metrics Summary Table


In [ ]:
models = eval_results['models']

print('=' * 90)
print(f'{"Model":<22} {"Type":<12} {"AUC-ROC":>10} {"F1":>10} {"Precision":>10} {"Recall":>10} {"FPR":>10}')
print('-' * 90)
for m in models:
    print(f'{m["name"]:<22} {m["type"]:<12} '
          f'{m["auc"]:>10.4f} '
          f'{m["f1"]:>10.4f} '
          f'{m["precision"]:>10.4f} '
          f'{m["recall"]:>10.4f} '
          f'{m["fpr"]:>10.4f}')
print('=' * 90)

# Color-coded summary
df_results = pd.DataFrame([{
    'Model': m['name'],
    'Type': m['type'],
    'AUC-ROC': m['auc'],
    'F1': m['f1'],
    'Precision': m['precision'],
    'Recall': m['recall'],
    'FPR': m['fpr'],
    'TP': m['tp'],
    'FP': m['fp'],
    'TN': m['tn'],
    'FN': m['fn']
} for m in models])

df_results = df_results.sort_values('AUC-ROC', ascending=False)
df_results


---

## 5. Performance Comparison Chart


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics = ['auc', 'f1', 'precision', 'recall']
titles  = ['AUC-ROC', 'F1 Score', 'Precision', 'Recall']
colors  = plt.cm.Set2(np.linspace(0, 1, len(models)))

for i, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[i]
    vals = [m[metric] for m in models]
    names = [m['name'] for m in models]
    bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Value labels
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')


---

## 6. Anomaly vs Supervised Analysis


In [ ]:
anomaly = [m for m in models if m['type'] == 'anomaly']
supervised = [m for m in models if m['type'] == 'supervised']

print('\nANOMALY-BASED MODELS (train on NORMAL only)')
print('-' * 60)
if anomaly:
    for m in sorted(anomaly, key=lambda x: -x['auc']):
        print(f'  {m["name"]:<25} AUC={m["auc"]:.4f}  F1={m["f1"]:.4f}  FPR={m["fpr"]:.4f}')
    
    print(f'\n  Average AUC: {np.mean([m["auc"] for m in anomaly]):.4f}')
    print(f'  Average F1:  {np.mean([m["f1"] for m in anomaly]):.4f}')
    print(f'  Average FPR: {np.mean([m["fpr"] for m in anomaly]):.4f}')

print('\nSUPERVISED MODELS (train on labeled data)')
print('-' * 60)
if supervised:
    for m in sorted(supervised, key=lambda x: -x['auc']):
        print(f'  {m["name"]:<25} AUC={m["auc"]:.4f}  F1={m["f1"]:.4f}  FPR={m["fpr"]:.4f}')
    
    print(f'\n  Average AUC: {np.mean([m["auc"] for m in supervised]):.4f}')
    print(f'  Average F1:  {np.mean([m["f1"] for m in supervised]):.4f}')
    print(f'  Average FPR: {np.mean([m["fpr"] for m in supervised]):.4f}')

# Comparison summary
print('\n' + '=' * 60)
print('ANALYSIS')
print('=' * 60)
if anomaly and supervised:
    best_anomaly   = max(anomaly, key=lambda x: x['auc'])
    best_supervised = max(supervised, key=lambda x: x['auc'])
    
    print(f'\n✓ Best anomaly model: {best_anomaly["name"]} (AUC={best_anomaly["auc"]:.4f})')
    print(f'✓ Best supervised model: {best_supervised["name"]} (AUC={best_supervised["auc"]:.4f})')
    
    auc_diff = best_supervised['auc'] - best_anomaly['auc']
    fpr_diff = best_anomaly['fpr'] - best_supervised['fpr']
    
    print(f'\n  Supervised AUC advantage: +{auc_diff:.4f}')
    print(f'  Anomaly FPR disadvantage: +{fpr_diff:.4f} (higher FPR)')
    
    print('\n  RECOMMENDATION:')
    print(f'  - Use {best_supervised["name"]} as PRIMARY detector (higher AUC)')
    print(f'  - Use {best_anomaly["name"]} as SECONDARY detector (zero-day detection)')
    print('  - Consider ensemble: alert only if BOTH models flag the same window')


## 7. Conclusions & Recommendations

### Key Findings

1. **Best model: CNN1D** với AUC=0.9423, F1=0.0033 (exceeds target of 0.90)
2. **BiLSTM** đạt AUC=0.9012 — cũng vượt target supervised (>0.90)
3. **Anomaly models** kém (AUC ~0.53) — Bot traffic quá giống Normal trong raw feature space
4. **FPR cao** (45%) — do extreme class imbalance (0.07% exfil trong test set)
5. **Precision thấp** — expected với imbalanced data, cần threshold tuning

### Deployment Recommendation

| Component | Model | Threshold | Purpose |
|---|---|---|---|
| Primary | CNN1D | prob > 0.8 | High-accuracy detection |
| Secondary | BiLSTM | prob > 0.8 | Ensemble partner |
| Heuristic | burst_exfil_score | > 0.7 | Fast pre-filter |

### For the Report

Chi tiết xem: `docs/metrics_report.md`

### Interpretation Note

F1 thấp (0.003) không phải thất bại — đây là **expected behavior** với extreme class imbalance (0.07% exfil). 
Metric quan trọng nhất là **AUC-ROC**: CNN1D đạt 0.9423 = **excellent discrimination**. 
AUC đo khả năng phân biệt giữa normal và exfil, không bị ảnh hưởng bởi class imbalance.
